# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:

import pandas as pd
import numpy as np
import faiss
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

# Memuat environment variables dari file .env
load_dotenv('../.env')

# Memeriksa apakah API key tersedia, tanpa menampilkan nilainya
api_key = os.environ.get('GEMINI_API_KEY')
print('Gemini API key tersedia:', bool(api_key))

client = genai.Client(api_key=api_key)

# Load knowledge base
df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')

print('Total entri knowledge base:', len(df))
print('Kolom:', df.columns.tolist())
print('Jumlah data kosong per kolom:')
print(df.isnull().sum())

df.head()


Gemini API key tersedia: True
Total entri knowledge base: 59
Kolom: ['faq_id', 'category', 'question', 'answer']
Jumlah data kosong per kolom:
faq_id      0
category    0
question    0
answer      0
dtype: int64


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [3]:
def get_embedding(text):
    result = client.models.embed_content(
        model='gemini-embedding-001',  # sesuaikan versi model embedding
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [4]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [5]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.33353508)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.4088098)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.4320181)}


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [6]:

def build_prompt(query, search_results):
    context = "\n\n".join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}"
        for r in search_results
    )

    prompt = f"""
Persona:
Kamu adalah asisten customer service e-commerce
yang ramah, jelas, dan membantu.

Task:
Jawab pertanyaan pengguna hanya berdasarkan konteks FAQ.
Jangan menambahkan fakta atau kebijakan yang tidak tersedia
dalam konteks.

Aturan menjawab:
1. Jika ada FAQ yang relevan, gunakan informasi tersebut
   untuk menjawab pertanyaan.
2. Jika jawaban memiliki kondisi tertentu, jelaskan kondisi
   tersebut dengan jelas.
3. Jika konteks hanya menjawab sebagian pertanyaan,
   sampaikan informasi yang tersedia dan jelaskan batasannya.
4. Jangan langsung mengatakan informasi tidak tersedia
   hanya karena pertanyaan menggunakan kata-kata berbeda
   dari FAQ.
5. Jika tidak ada FAQ yang relevan, katakan bahwa informasi
   belum tersedia dan sarankan menghubungi customer service.

Context:
{context}

Pertanyaan pengguna:
{query}

Format:
Berikan jawaban dalam bahasa Indonesia yang ringkas,
ramah, dan mudah dipahami.
"""

    return prompt


In [7]:

query = 'kenapa paket saya belum sampai?'

search_results = semantic_search(query, top_k=3)

prompt = build_prompt(query, search_results)

print(prompt)



Persona:
Kamu adalah asisten customer service e-commerce
yang ramah, jelas, dan membantu.

Task:
Jawab pertanyaan pengguna hanya berdasarkan konteks FAQ.
Jangan menambahkan fakta atau kebijakan yang tidak tersedia
dalam konteks.

Aturan menjawab:
1. Jika ada FAQ yang relevan, gunakan informasi tersebut
   untuk menjawab pertanyaan.
2. Jika jawaban memiliki kondisi tertentu, jelaskan kondisi
   tersebut dengan jelas.
3. Jika konteks hanya menjawab sebagian pertanyaan,
   sampaikan informasi yang tersedia dan jelaskan batasannya.
4. Jangan langsung mengatakan informasi tidak tersedia
   hanya karena pertanyaan menggunakan kata-kata berbeda
   dari FAQ.
5. Jika tidak ada FAQ yang relevan, katakan bahwa informasi
   belum tersedia dan sarankan menghubungi customer service.

Context:
FAQ: Bagaimana jika paket saya belum sampai melebihi estimasi?
Jawaban: Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusur

## 7. Mengintegrasikan LLM

In [8]:
available_models = []

for model in client.models.list():
    if 'generateContent' in (model.supported_actions or []):
        available_models.append(model.name)
        print(model.name)

print(f"\nTotal model yang mendukung generateContent: {len(available_models)}")

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

In [9]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',  # sesuaikan versi model
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Jawaban LLM:
Halo! Tentu, Anda bisa membayar dengan cicilan. Fitur ini tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan oleh pihak bank ya. Ada hal lain yang bisa saya bantu?

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [12]:
for q in test_queries:
    print("Query:", q)
    answer, sources = generate_answer(q)

    print("Jawaban:", answer)
    print("Sumber FAQ:")

    for s in sources:
        print(
            f"- {s['question']} "
            f"(distance: {s['distance']:.4f})"
        )

    print("---")

Query: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Halo! Tentu, saya akan bantu jelaskan ya. 😊

Cara melakukan pemesanan di platform ini sangat mudah:
1. Pilih produk yang Anda inginkan.
2. Klik tombol 'Tambah ke Keranjang'.
3. Lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan memilih metode pembayaran.

Sebagai informasi, saat ini pemesanan hanya dapat dilakukan setelah Anda membuat akun ya, agar Anda bisa melacak status pesanan dengan mudah. 

Ada hal lain yang ingin ditanyakan?
Sumber FAQ:
- Bagaimana cara melakukan pemesanan di platform ini? (distance: 0.2844)
- Apakah saya bisa memesan tanpa membuat akun? (distance: 0.5036)
- Bagaimana cara mengetahui status pesanan saya saat ini? (distance: 0.5704)
---
Query: Bisakah saya membatalkan order yang sudah saya buat?
Jawaban: Halo! Tentu, Anda bisa membatalkan pesanan yang sudah dibuat melalui halaman 'Pesanan Saya'. 

Namun, ada ketentuannya ya: pembatalan hanya dapat dilakukan selama status pesanan ma

In [11]:

test_queries = [
    # 1. Jawaban jelas tersedia di FAQ
    "Bagaimana cara melakukan pemesanan di platform ini?",

    # 2. Jawaban jelas tersedia, tetapi kalimatnya berbeda
    "Bisakah saya membatalkan order yang sudah saya buat?",

    # 3. Parafrasa: masalah keterlambatan pengiriman
    "Kenapa paket saya belum sampai?",

    # 4. Parafrasa: masalah pembayaran cicilan
    "Apakah saya bisa membayar belanjaan secara angsuran?",

    # 5. Pertanyaan yang kemungkinan tidak tercakup dalam FAQ
    "Apakah toko menyediakan layanan pemasangan produk di rumah?"
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')


Query: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Halo! Tentu, saya akan bantu jelaskan ya. 

Untuk melakukan pemesanan, caranya sangat mudah:
1. Pilih produk yang Anda inginkan.
2. Klik tombol 'Tambah ke Keranjang'.
3. Lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan memilih metode pembayaran.

*Catatan: Saat ini pemesanan hanya dapat dilakukan setelah Anda membuat akun ya, agar Anda bisa melacak status pesanan dengan mudah.*

Ada lagi yang bisa saya bantu?
Sumber teratas: Bagaimana cara melakukan pemesanan di platform ini?
---
Query: Bisakah saya membatalkan order yang sudah saya buat?
Jawaban: Halo! Tentu, Anda bisa membatalkan pesanan yang sudah dibuat melalui halaman 'Pesanan Saya' selama statusnya masih 'Diproses'. Namun, perlu diingat bahwa pembatalan tidak dapat dilakukan jika status pesanan sudah berubah menjadi 'Dikirim'. 

Ada lagi yang bisa saya bantu?
Sumber teratas: Bagaimana cara membatalkan pesanan yang sudah dibuat?
---
Query: Kenapa 

### Catatan Analisis Relevansi

Berdasarkan pengujian terhadap lima query, sistem Semantic Search menunjukkan hasil retrieval yang baik pada empat query pertama dan hasil yang kurang relevan pada query kelima.

**1. Query dengan jawaban yang tersedia di knowledge base**

Pada query “Bagaimana cara melakukan pemesanan di platform ini?”, sistem berhasil menemukan FAQ yang sesuai pada peringkat pertama. Hasil yang sama juga terlihat pada query “Bisakah saya membatalkan order yang sudah saya buat?”, yang berhasil menemukan FAQ tentang pembatalan pesanan meskipun menggunakan kata *order* sebagai pengganti *pesanan*. Kedua hasil ini menunjukkan bahwa sistem dapat menemukan informasi yang relevan, baik pada pertanyaan yang sama maupun pertanyaan dengan parafrasa.

**2. Query dengan parafrasa atau sinonim**

Pada query “Apakah saya bisa membayar belanjaan secara angsuran?”, sistem berhasil menemukan FAQ tentang pembayaran cicilan pada peringkat pertama. Hal ini menunjukkan bahwa semantic search dapat mengenali hubungan makna antara kata *angsuran* dan *cicilan*, meskipun kata yang digunakan berbeda dari FAQ asli.

**3. Query mengenai keterlambatan pengiriman**

Pada query “Kenapa paket saya belum sampai?”, sistem menemukan FAQ tentang paket yang terlambat melebihi estimasi pada peringkat pertama, diikuti FAQ mengenai status pengiriman yang tidak berubah dan nomor resi yang belum dapat dilacak. Ketiga hasil tersebut relevan dengan permasalahan pengiriman.

Pada pengujian awal, LLM menyatakan bahwa informasi belum tersedia meskipun konteks yang relevan telah ditemukan. Setelah prompt diperbaiki agar model menggunakan FAQ yang relevan, menjelaskan kondisi tertentu, dan menjawab berdasarkan informasi yang tersedia, LLM menghasilkan jawaban yang lebih sesuai. Jawaban tersebut menjelaskan bahwa pengguna dapat menghubungi customer service apabila keterlambatan melebihi tiga hari dari estimasi pengiriman, dengan menyertakan nomor resi.

**4. Query di luar cakupan knowledge base**

Pada query “Apakah toko menyediakan layanan pemasangan produk di rumah?”, hasil retrieval teratas adalah FAQ tentang permintaan produk kustom, diikuti FAQ mengenai pengiriman same-day dan perubahan alamat pengiriman. Hasil tersebut tidak menjawab pertanyaan pengguna secara langsung sehingga relevansi retrieval dinilai kurang baik.

Meskipun demikian, LLM berhasil menyatakan bahwa informasi mengenai layanan pemasangan produk di rumah belum tersedia dan menyarankan pengguna menghubungi customer service. Respons ini menunjukkan bahwa prompt membantu membatasi jawaban agar tidak mengarang informasi yang tidak didukung oleh knowledge base.

**Kesimpulan**

Secara keseluruhan, sistem berhasil mengambil FAQ yang relevan untuk empat dari lima query yang diuji, termasuk pertanyaan dengan parafrasa atau sinonim. Perbaikan prompt juga meningkatkan kemampuan LLM dalam memanfaatkan konteks FAQ dan menangani pertanyaan yang tidak memiliki jawaban langsung di knowledge base.

Keterbatasan yang ditemukan adalah semantic search tetap mengembalikan hasil teratas meskipun relevansinya rendah. Selain itu, kualitas jawaban LLM bergantung pada relevansi konteks retrieval dan kejelasan instruksi prompt. Pengujian ini masih terbatas pada lima query sehingga belum mewakili seluruh variasi pertanyaan pengguna.

In [34]:

import pandas as pd
from IPython.display import display

evaluation_data = [
    {
        "No": 1,
        "Query": "Bagaimana cara melakukan pemesanan di platform ini?",
        "FAQ yang seharusnya relevan": "ID 1 — Cara melakukan pemesanan",
        "Evaluasi": (
            "Baik. Retrieval menemukan FAQ yang tepat pada peringkat "
            "pertama dan jawaban LLM sesuai dengan informasi yang tersedia."
        )
    },
    {
        "No": 2,
        "Query": "Bisakah saya membatalkan order yang sudah saya buat?",
        "FAQ yang seharusnya relevan": "ID 3 — Cara membatalkan pesanan",
        "Evaluasi": (
            "Baik. Sistem memahami parafrasa order dan pesanan. "
            "FAQ relevan ditemukan pada peringkat pertama dan jawaban "
            "LLM sesuai dengan konteks."
        )
    },
    {
        "No": 3,
        "Query": "Kenapa paket saya belum sampai?",
        "FAQ yang seharusnya relevan": (
            "ID 12 — Paket belum sampai melebihi estimasi; "
            "ID 14 — Status pengiriman tidak berubah; "
            "ID 52 — Nomor resi belum bisa dilacak"
        ),
        "Evaluasi": (
            "Cukup baik. Retrieval menemukan beberapa FAQ relevan. "
            "Pada pengujian awal, LLM belum memanfaatkan konteks dengan "
            "tepat. Setelah prompt diperbaiki, jawaban menjadi sesuai "
            "dengan informasi FAQ."
        )
    },
    {
        "No": 4,
        "Query": "Apakah saya bisa membayar belanjaan secara angsuran?",
        "FAQ yang seharusnya relevan": "ID 20 — Pembayaran cicilan",
        "Evaluasi": (
            "Baik. Sistem mengenali hubungan makna angsuran dan cicilan. "
            "FAQ ditemukan pada peringkat pertama dan jawaban LLM sesuai."
        )
    },
    {
        "No": 5,
        "Query": "Apakah toko menyediakan layanan pemasangan produk di rumah?",
        "FAQ yang seharusnya relevan": (
            "Tidak ditemukan FAQ khusus tentang pemasangan produk di rumah."
        ),
        "Evaluasi": (
            "Baik dalam menjaga batasan informasi, tetapi retrieval "
            "kurang relevan. LLM tidak mengarang kebijakan dan menyarankan "
            "pengguna menghubungi customer service."
        )
    }
]

df_evaluation = pd.DataFrame(evaluation_data)

display(df_evaluation)


print("KESIMPULAN EVALUASI")
print(
    "Dari lima query yang diuji, empat query berhasil menemukan "
    "FAQ yang relevan pada peringkat pertama, sedangkan satu query "
    "menunjukkan hasil retrieval yang kurang relevan."
)
print(
    "Perbaikan prompt membantu LLM memanfaatkan konteks FAQ dengan "
    "lebih tepat dan menghindari jawaban yang tidak didukung oleh "
    "knowledge base."
)
print(
    "Evaluasi ini bersifat manual dan terbatas pada lima query uji."
)



,No,Query,FAQ yang seharusnya relevan,Evaluasi
0,1,Bagaimana cara melakukan pemesanan di platform ini?,ID 1 — Cara melakukan pemesanan,Baik. Retrieval menemukan FAQ yang tepat pada peringkat pertama dan jawaban LLM sesuai dengan informasi yang tersedia.
1,2,Bisakah saya membatalkan order yang sudah saya buat?,ID 3 — Cara membatalkan pesanan,Baik. Sistem memahami parafrasa order dan pesanan. FAQ relevan ditemukan pada peringkat pertama dan jawaban LLM sesuai dengan konteks.
2,3,Kenapa paket saya belum sampai?,ID 12 — Paket belum sampai melebihi estimasi; ID 14 — Status pengiriman tidak berubah; ID 52 — Nomor resi belum bisa dilacak,"Cukup baik. Retrieval menemukan beberapa FAQ relevan. Pada pengujian awal, LLM belum memanfaatkan konteks dengan tepat. Setelah prompt diperbaiki,..."
3,4,Apakah saya bisa membayar belanjaan secara angsuran?,ID 20 — Pembayaran cicilan,Baik. Sistem mengenali hubungan makna angsuran dan cicilan. FAQ ditemukan pada peringkat pertama dan jawaban LLM sesuai.
4,5,Apakah toko menyediakan layanan pemasangan produk di rumah?,Tidak ditemukan FAQ khusus tentang pemasangan produk di rumah.,"Baik dalam menjaga batasan informasi, tetapi retrieval kurang relevan. LLM tidak mengarang kebijakan dan menyarankan pengguna menghubungi customer..."


KESIMPULAN EVALUASI
Dari lima query yang diuji, empat query berhasil menemukan FAQ yang relevan pada peringkat pertama, sedangkan satu query menunjukkan hasil retrieval yang kurang relevan.
Perbaikan prompt membantu LLM memanfaatkan konteks FAQ dengan lebih tepat dan menghindari jawaban yang tidak didukung oleh knowledge base.
Evaluasi ini bersifat manual dan terbatas pada lima query uji.


In [35]:

import pandas as pd

retrieval_rows = []

for query in test_queries:
    results = semantic_search(query, top_k=3)

    for result in results:
        retrieval_rows.append({
            "Query": query,
            "Rank": result["rank"],
            "FAQ": result["question"],
            "Kategori": result["category"],
            "Distance": round(float(result["distance"]), 4),
            "Jawaban FAQ": result["answer"]
        })

df_retrieval_results = pd.DataFrame(retrieval_rows)

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.max_rows", 20)

display(df_retrieval_results)


,Query,Rank,FAQ,Kategori,Distance,Jawaban FAQ
0,Bagaimana cara melakukan pemesanan di platform ini?,1,Bagaimana cara melakukan pemesanan di platform ini?,Pemesanan,0.2844,"Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran."
1,Bagaimana cara melakukan pemesanan di platform ini?,2,Apakah saya bisa memesan tanpa membuat akun?,Pemesanan,0.5036,"Saat ini pemesanan hanya bisa dilakukan setelah membuat akun, agar Anda dapat melacak status pesanan dengan mudah."
2,Bagaimana cara melakukan pemesanan di platform ini?,3,Bagaimana cara mengetahui status pesanan saya saat ini?,Pemesanan,0.5704,Anda bisa memeriksa status pesanan secara real-time melalui menu 'Pesanan Saya' di akun Anda.
3,Bisakah saya membatalkan order yang sudah saya buat?,1,Bagaimana cara membatalkan pesanan yang sudah dibuat?,Pemesanan,0.2329,"Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama status masih 'Diproses'. Setelah status berubah menjadi 'Dikirim', pembatalan tidak..."
4,Bisakah saya membatalkan order yang sudah saya buat?,2,Bagaimana prosedur pengembalian barang yang tidak sesuai?,Pengembalian,0.4842,"Ajukan pengembalian melalui menu 'Pesanan Saya' dengan memilih alasan dan menyertakan bukti foto, maksimal 3 hari sejak barang diterima."
5,Bisakah saya membatalkan order yang sudah saya buat?,3,Bagaimana jika produk yang saya pesan ternyata kehabisan stok?,Produk,0.4934,"Jika stok habis setelah pesanan dibuat, penjual akan membatalkan sebagian/seluruh pesanan dan dana otomatis dikembalikan."
6,Kenapa paket saya belum sampai?,1,Bagaimana jika paket saya belum sampai melebihi estimasi?,Pengiriman,0.3421,"Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih l..."
7,Kenapa paket saya belum sampai?,2,Kenapa status pengiriman saya tidak berubah selama beberapa hari?,Pengiriman,0.4070,Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.
8,Kenapa paket saya belum sampai?,3,Kenapa nomor resi saya belum bisa dilacak di website kurir?,Pengiriman,0.4336,"Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'."
9,Apakah saya bisa membayar belanjaan secara angsuran?,1,Apakah saya bisa membayar dengan cicilan?,Pembayaran,0.3336,Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.


In [36]:

diagnostic_queries = [
    "Kenapa paket saya belum sampai?",
    "Apakah toko menyediakan layanan pemasangan produk di rumah?"
]

for q in diagnostic_queries:
    print("\n" + "=" * 70)
    print("QUERY:", q)

    results = semantic_search(q, top_k=3)

    print("\nHASIL RETRIEVAL:")
    for r in results:
        print(f"\nRank: {r['rank']}")
        print(f"FAQ: {r['question']}")
        print(f"Kategori: {r['category']}")
        print(f"Distance: {float(r['distance']):.4f}")
        print(f"Jawaban: {r['answer']}")

    print("\nPROMPT YANG DIKIRIM KE LLM:")
    print(build_prompt(q, results))



QUERY: Kenapa paket saya belum sampai?

HASIL RETRIEVAL:

Rank: 1
FAQ: Bagaimana jika paket saya belum sampai melebihi estimasi?
Kategori: Pengiriman
Distance: 0.3421
Jawaban: Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.

Rank: 2
FAQ: Kenapa status pengiriman saya tidak berubah selama beberapa hari?
Kategori: Pengiriman
Distance: 0.4070
Jawaban: Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.

Rank: 3
FAQ: Kenapa nomor resi saya belum bisa dilacak di website kurir?
Kategori: Pengiriman
Distance: 0.4336
Jawaban: Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.

PROMPT YANG DIKIRIM KE LLM:

Persona:
Kamu adalah asisten customer service e-commerce
yang ramah, jelas, dan membantu.

Task:
Jawab pertan

In [13]:

answer, sources = generate_answer(
    "Apakah saya bisa membayar belanjaan secara angsuran?"

)

print("Jawaban LLM:")
print(answer)

print("\nSumber FAQ yang digunakan:")
for s in sources:
    print(f"- {s['question']}")


Jawaban LLM:
Halo! Tentu, Anda bisa membayar belanjaan secara angsuran (cicilan). 

Fitur cicilan ini tersedia untuk transaksi dengan kartu kredit tertentu, ya, dengan nominal minimal yang telah ditentukan oleh pihak bank. 

Ada lagi yang bisa saya bantu hari ini?

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Bagaimana cara melakukan pemesanan di platform ini?


In [32]:
answer, sources = generate_answer(
    "Apakah toko menyediakan layanan pemasangan produk di rumah?"
)

print("Jawaban LLM:")
print(answer)

print("\nSumber FAQ yang digunakan:")
for s in sources:
    print(f"- {s['question']}")

Jawaban LLM:
Halo! Mohon maaf, informasi mengenai layanan pemasangan produk di rumah saat ini belum tersedia dalam sistem kami. 

Jika Anda membutuhkan bantuan lebih lanjut, silakan menghubungi customer service kami ya. Terima kasih!

Sumber FAQ yang digunakan:
- Apakah saya bisa request custom produk kepada penjual?
- Apakah tersedia opsi pengiriman same-day?
- Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?
